# LLM Preference Prediction with DistilBERT (Kaggle-ready)

This notebook trains a DistilBERT classifier for the Chatbot Arena preference task and creates `submission.csv`.

Pipeline:
- Robust parsing of list-formatted fields (`prompt`, `response_a`, `response_b`)
- EDA + sequence-length analysis
- DistilBERT fine-tuning with strong Kaggle settings
- Response-swap TTA at inference to reduce position bias
- Competition-format submission file


In [ ]:
import ast
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

warnings.filterwarnings('ignore')

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
print('CUDA available:', torch.cuda.is_available())


In [ ]:
KAGGLE_DATA_DIR = Path('/kaggle/input/competitions/llm-classification-finetuning')
LOCAL_DATA_DIR = Path('./data')

DATA_DIR = KAGGLE_DATA_DIR if KAGGLE_DATA_DIR.exists() else LOCAL_DATA_DIR
print('Using DATA_DIR:', DATA_DIR)

train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print('train shape:', train_df.shape)
print('test shape :', test_df.shape)
train_df.head(2)


In [ ]:
LABEL_COLS = ['winner_model_a', 'winner_model_b', 'winner_tie']

def parse_turns(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return [str(t) for t in x]

    s = str(x).strip()
    if not s:
        return []

    for parser in (ast.literal_eval, json.loads):
        try:
            val = parser(s)
            if isinstance(val, list):
                return [str(t) for t in val]
            return [str(val)]
        except Exception:
            pass

    return [s]

def sanitize_text(s):
    s = str(s)
    return s.encode('utf-8', errors='replace').decode('utf-8')

def normalize_text(s):
    return ' '.join(sanitize_text(s).replace('\n', ' ').split())

def join_turns(x):
    turns = [normalize_text(t) for t in parse_turns(x)]
    turns = [t for t in turns if t]
    return ' [TURN] '.join(turns)

def build_input(prompt, response_a, response_b):
    p = join_turns(prompt)
    a = join_turns(response_a)
    b = join_turns(response_b)
    return sanitize_text(f'[PROMPT] {p} [SEP] [RESPONSE_A] {a} [SEP] [RESPONSE_B] {b}')

train_df['text'] = train_df.apply(
    lambda r: build_input(r['prompt'], r['response_a'], r['response_b']), axis=1
)
test_df['text'] = test_df.apply(
    lambda r: build_input(r['prompt'], r['response_a'], r['response_b']), axis=1
)
test_df['text_swapped'] = test_df.apply(
    lambda r: build_input(r['prompt'], r['response_b'], r['response_a']), axis=1
)

train_df['label'] = train_df[LABEL_COLS].values.argmax(axis=1).astype(int)

print('Class distribution:')
print(train_df['label'].value_counts(normalize=True).sort_index())
train_df[['id', 'text', 'label']].head(2)

In [ ]:
train_df['char_len'] = train_df['text'].str.len()
train_df['word_len'] = train_df['text'].str.split().str.len()

print(train_df[['char_len', 'word_len']].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

p95_words = int(train_df['word_len'].quantile(0.95))
MAX_LENGTH = int(np.clip(p95_words * 1.6, 192, 512))
print(f'MAX_LENGTH selected: {MAX_LENGTH} (based on p95 words={p95_words})')


In [ ]:
MODEL_CANDIDATES = [
    '/kaggle/input/distilbert-base-uncased',
    'distilbert-base-uncased',
]
MODEL_NAME = MODEL_CANDIDATES[0] if Path(MODEL_CANDIDATES[0]).exists() else MODEL_CANDIDATES[1]
print('Using MODEL_NAME:', MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_split_df, valid_split_df = train_test_split(
    train_df[['id', 'text', 'label']],
    test_size=0.1,
    random_state=42,
    stratify=train_df['label'],
)

train_ds = Dataset.from_pandas(train_split_df.reset_index(drop=True))
valid_ds = Dataset.from_pandas(valid_split_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df[['id', 'text']].reset_index(drop=True))
test_ds_swapped = Dataset.from_pandas(
    test_df[['id', 'text_swapped']].rename(columns={'text_swapped': 'text'}).reset_index(drop=True)
)

def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)

train_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=['id', 'text'])
valid_ds = valid_ds.map(tokenize_fn, batched=True, remove_columns=['id', 'text'])
test_ds = test_ds.map(tokenize_fn, batched=True, remove_columns=['id', 'text'])
test_ds_swapped = test_ds_swapped.map(tokenize_fn, batched=True, remove_columns=['id', 'text'])

train_ds = train_ds.rename_column('label', 'labels')
valid_ds = valid_ds.rename_column('label', 'labels')

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None
)

print('Train split:', len(train_ds), '| Valid split:', len(valid_ds))


In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=train_split_df['label'].values
)
class_weights = torch.tensor(class_weights, dtype=torch.float32)
print('class_weights:', class_weights.tolist())

id2label = {0: 'winner_model_a', 1: 'winner_model_b', 2: 'winner_tie'}
label2id = {v: k for k, v in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)
print('Note: missing classifier weights are expected here because the classification head is newly initialized for this task.')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = probs.argmax(axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'log_loss': log_loss(labels, probs, labels=[0, 1, 2])
    }

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.model_accepts_loss_kwargs = False

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.get('logits')
        cw = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss = F.cross_entropy(logits, labels, weight=cw)
        return (loss, outputs) if return_outputs else loss

output_dir = '/kaggle/working/distilbert_preference_model' if Path('/kaggle').exists() else './trained_models/distilbert_preference_model'
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

training_args = TrainingArguments(
    output_dir=output_dir,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model='log_loss',
    greater_is_better=False,
    bf16=use_bf16,
    fp16=use_fp16,
    gradient_checkpointing=True,
    report_to='none',
    save_total_limit=2,
    save_only_model=True,
    seed=42
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
    class_weights=class_weights
)

trainer.train()
print(trainer.evaluate())

In [ ]:
def softmax_np(x):
    x = x - np.max(x, axis=1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)

base_logits = trainer.predict(test_ds).predictions
base_probs = softmax_np(base_logits)

swap_logits = trainer.predict(test_ds_swapped).predictions
swap_probs = softmax_np(swap_logits)
swap_probs_back = swap_probs[:, [1, 0, 2]]

final_probs = 0.5 * base_probs + 0.5 * swap_probs_back

submission = pd.DataFrame({
    'id': test_df['id'].values,
    'winner_model_a': final_probs[:, 0],
    'winner_model_b': final_probs[:, 1],
    'winner_tie': final_probs[:, 2],
})

submission.to_csv('submission.csv', index=False)
print('Saved submission.csv:', submission.shape)
submission.head()


In [ ]:
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print('Saved model/tokenizer to', output_dir)
